In [ ]:
import pandas as pd
import nltk
import string
import math
import re 
from tqdm.auto import tqdm
from nltk import ngrams
from scipy.stats import kendalltau


df_moral          = pd.read_csv("./data/brmoral/final_brmoral.csv", sep="\t")




/home/pablo/miniconda3/envs/unsloth/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
"""String similarity metrics"""

def get_size(x):
    return len(nltk.tokenize.word_tokenize(x, language='portuguese', preserve_line=False))


def tokenize(text, nGram=3):
    text = text.split(" ")
    grams = []
    for i in range(1, nGram):
        i_grams = [
            " ".join(gram)
            for gram in ngrams(text, i)
        ]
        grams.extend(i_grams)
        
    return grams

def getNgramOverlap(x, nGram, source, target):
  overlaps = []
  for h, r in zip([x[source]], [x[target]]):
    if (h == "") or (r == ""):
      overlaps.append(1.0)
      continue
    a = tokenize(h, nGram)
    b = tokenize(r, nGram)

    if len(a) >= len(b) and len(a) > 0:
      overlaps.append(len(set(a) & set(b))/len(a))
    elif len(b) >= len(a) and len(b) > 0:
      overlaps.append(len(set(a) & set(b))/len(b))
    elif len(a) == 0 or len(b) == 0:
      overlaps.append(0)

  return overlaps[0]



def get_kendall_tau(x, source, target):
    x1 = normalize_answer(x[source])
    x2 = normalize_answer(x[target])

    x1_tokens = x1.split()
    x2_tokens = x2.split()

    for x1_index, tok in enumerate(x1_tokens):
        try:
            x2_index = x2_tokens.index(tok)
            x1_tokens[x1_index] = "<match-found>-{:d}".format(x1_index + 1)
            x2_tokens[x2_index] = "<match-found>-{:d}".format(x1_index + 1)
        except ValueError:
            pass

    common_seq_x1 = [int(x1_tok_flag.split("-")[-1]) for x1_tok_flag in x1_tokens if x1_tok_flag.startswith("<match-found>")]
    common_seq_x2 = [int(x2_tok_flag.split("-")[-1]) for x2_tok_flag in x2_tokens if x2_tok_flag.startswith("<match-found>")]

    assert len(common_seq_x1) == len(common_seq_x2)

    ktd = kendalltau(common_seq_x1, common_seq_x2).correlation
    anomaly = False

    if math.isnan(ktd):
        ktd = -1.0
        anomaly = True

    return ktd


def normalize_answer(s):
    """Lower text and remove punctuation and extra whitespace."""


    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_punc(lower(s)))



In [ ]:
"""Overlap between the author original text and the gerneric version of the text (OpenAI)"""
df_moral["gay-marriage-overlap"] = df_moral.apply(lambda x: getNgramOverlap(x, 3, "t.gay-marriage", "open.gay-marriage"), axis=1)
df_moral["gun-control-overlap"] = df_moral.apply(lambda x: getNgramOverlap(x, 3, "t.gun-control", "open.gun-control"), axis=1)
df_moral["abortion-overlap"] = df_moral.apply(lambda x: getNgramOverlap(x, 3, "t.abortion", "open.abortion"), axis=1)
df_moral["death-penalty-overlap"] = df_moral.apply(lambda x: getNgramOverlap(x, 3, "t.death-penalty", "open.death-penalty"), axis=1)
df_moral["drugs-overlap"] = df_moral.apply(lambda x: getNgramOverlap(x, 3, "t.drugs", "open.drugs"), axis=1)
df_moral["criminal-age-overlap"] = df_moral.apply(lambda x: getNgramOverlap(x, 3, "t.criminal-age", "open.criminal-age"), axis=1)
df_moral["racial-quotas-overlap"] = df_moral.apply(lambda x: getNgramOverlap(x, 3, "t.racial-quotas", "open.racial-quotas"), axis=1)
df_moral["church-tax-overlap"] = df_moral.apply(lambda x: getNgramOverlap(x, 3, "t.church-tax", "open.church-tax"), axis=1)


print(df_moral["gay-marriage-overlap"].mean())
print(df_moral["gun-control-overlap"].mean())
print(df_moral["abortion-overlap"].mean())
print(df_moral["death-penalty-overlap"].mean())
print(df_moral["drugs-overlap"].mean())
print(df_moral["criminal-age-overlap"].mean())
print(df_moral["racial-quotas-overlap"].mean())
print(df_moral["church-tax-overlap"].mean())



0.349434263587006
0.312139425340889
0.3157408607124405
0.3032543947415138
0.3147173868934046
0.3113512096223469
0.3052779350626886
0.3055337146403123


In [ ]:
"""GET THE SIZE OF THE TEXT"""
print("Orignal text size:\n")

df_moral["original-size-gay-marriage"] = df_moral.apply(lambda x: get_size(x["t.gay-marriage"]), axis=1)
df_moral["original-size-gun-control"] = df_moral.apply(lambda x: get_size(x["t.gun-control"]), axis=1)
df_moral["original-size-abortion"] = df_moral.apply(lambda x: get_size(x["t.abortion"]), axis=1)
df_moral["original-size-death-penalty"] = df_moral.apply(lambda x: get_size(x["t.death-penalty"]), axis=1)
df_moral["original-size-drugs"] = df_moral.apply(lambda x: get_size(x["t.drugs"]), axis=1)
df_moral["original-size-criminal-age"] = df_moral.apply(lambda x: get_size(x["t.criminal-age"]), axis=1)
df_moral["original-size-racial-quotas"] = df_moral.apply(lambda x: get_size(x["t.racial-quotas"]), axis=1)
df_moral["original-size-church-tax"] = df_moral.apply(lambda x: get_size(x["t.church-tax"]), axis=1)

print(df_moral["original-size-gay-marriage"].mean())
print(df_moral["original-size-gun-control"].mean())
print(df_moral["original-size-abortion"].mean())
print(df_moral["original-size-death-penalty"].mean())
print(df_moral["original-size-drugs"].mean())
print(df_moral["original-size-criminal-age"].mean())
print(df_moral["original-size-racial-quotas"].mean())
print(df_moral["original-size-church-tax"].mean())

print("----------------------")
print("OpenAI text size:\n")    

df_moral["open-size-gay-marriage"] = df_moral.apply(lambda x: get_size(x["open.gay-marriage"]), axis=1)
df_moral["open-size-gun-control"] = df_moral.apply(lambda x: get_size(x["open.gun-control"]), axis=1)
df_moral["open-size-abortion"] = df_moral.apply(lambda x: get_size(x["open.abortion"]), axis=1)
df_moral["open-size-death-penalty"] = df_moral.apply(lambda x: get_size(x["open.death-penalty"]), axis=1)
df_moral["open-size-drugs"] = df_moral.apply(lambda x: get_size(x["open.drugs"]), axis=1)
df_moral["open-size-criminal-age"] = df_moral.apply(lambda x: get_size(x["open.criminal-age"]), axis=1)
df_moral["open-size-racial-quotas"] = df_moral.apply(lambda x: get_size(x["open.racial-quotas"]), axis=1)
df_moral["open-size-church-tax"] = df_moral.apply(lambda x: get_size(x["open.church-tax"]), axis=1)


print(df_moral["open-size-gay-marriage"].mean())
print(df_moral["open-size-gun-control"].mean())
print(df_moral["open-size-abortion"].mean())
print(df_moral["open-size-death-penalty"].mean())
print(df_moral["open-size-drugs"].mean())
print(df_moral["open-size-criminal-age"].mean())
print(df_moral["open-size-racial-quotas"].mean())
print(df_moral["open-size-church-tax"].mean())



Orignal text size:

51.64509803921569
61.29019607843137
68.24901960784314
59.23725490196078
60.258823529411764
58.93921568627451
64.69019607843137
50.84117647058824
----------------------
OpenAI text size:

51.24901960784314
62.25098039215686
68.87254901960785
59.15098039215686
61.87843137254902
59.80980392156863
65.87450980392157
52.043137254901964
----------------------


In [20]:
"""Kendau-tau between the author original text and the gerneric version of the text (OpenAI)"""

df_moral["gay-marriage-kendall"] = df_moral.apply(lambda x: get_kendall_tau(x, "t.gay-marriage", "open.gay-marriage"), axis=1)
df_moral["gun-control-kendall"] = df_moral.apply(lambda x: get_kendall_tau(x, "t.gun-control", "open.gun-control"), axis=1)
df_moral["abortion-kendall"] = df_moral.apply(lambda x: get_kendall_tau(x, "t.abortion", "open.abortion"), axis=1)
df_moral["death-penalty-kendall"] = df_moral.apply(lambda x: get_kendall_tau(x, "t.death-penalty", "open.death-penalty"), axis=1)
df_moral["drugs-kendall"] = df_moral.apply(lambda x: get_kendall_tau(x, "t.drugs", "open.drugs"), axis=1)
df_moral["criminal-age-kendall"] = df_moral.apply(lambda x: get_kendall_tau(x, "t.criminal-age", "open.criminal-age"), axis=1)
df_moral["racial-quotas-kendall"] = df_moral.apply(lambda x: get_kendall_tau(x, "t.racial-quotas", "open.racial-quotas"), axis=1)
df_moral["church-tax-kendall"] = df_moral.apply(lambda x: get_kendall_tau(x, "t.church-tax", "open.church-tax"), axis=1)


print(df_moral["gay-marriage-kendall"].mean())
print(df_moral["gun-control-kendall"].mean())
print(df_moral["abortion-kendall"].mean())
print(df_moral["death-penalty-kendall"].mean())
print(df_moral["drugs-kendall"].mean())
print(df_moral["criminal-age-kendall"].mean())
print(df_moral["racial-quotas-kendall"].mean())
print(df_moral["church-tax-kendall"].mean())

0.7861560398405988
0.7452938396143866
0.7514892602558358
0.7627313588600296
0.7695935580761779
0.7675127013667835
0.7635126609393521
0.7845318292599283


In [ ]:
df_ustance_cino   = pd.read_csv("./data/ustance/final_cino.csv", sep="\t")
df_ustance_cl     = pd.read_csv("./data/ustance/final_cl.csv", sep="\t")
df_ustance_igreja = pd.read_csv("./data/ustance/final_igreja.csv", sep="\t")


In [37]:
df_ustance_cino

,Text,Target,Polarity,Factual,User_Name,User_ID,Tweet_ID,Tweet_Seq,POS,Mentions,Nouns,generated_Text
0,"Noblatzão, esse Dória é um Fanfarrão.... prime...",r2_co,against,no,bittencourtleon,r2_co_2461,_1298936590115508225,2052,"[('@', 'NUM'), ('blogdonoblat', 'N'), ('noblat...",['blogdonoblat'],"{'vacina': 3, 'bilhões': 1, 'dória': 1, '2021'...","Noblatzão, esse Dória é um brincalhão... prime..."
1,Esse pastor bem que merece um processo. Pasto...,r2_co,for,no,bimaldo,r2_co_2452,_1338529405497470982,1736,"[('esse', 'PROADJ'), ('pastor', 'N'), ('bem', ...",[],"{'pastor': 2, 'processo': 1, 'câncer': 1, 'hiv...",Esse pastor certamente merece enfrentar um pro...
2,"O Ministro da Saúde diz: ""O Brasil vai comprar...",r2_co,for,no,alxwatanabeph,r2_co_1953,_1318944182124351488,2478,"[('o', 'ART'), ('ministro', 'N'), ('da', 'PREP...",[],"{'vacina': 2, 'bolsonaro': 1, 'milhões': 1, 'm...","O Ministro da Saúde declara: ""O Brasil adquiri..."
3,Kim K já tomou a coronavac e tá virando jacaré,r2_co,for,no,cahrolfagundes,r2_co_2672,_1342837383876767745,1537,"[('kim', 'N'), ('k', 'N'), ('já', 'ADV'), ('to...",[],"{'kim': 1, 'coronavac': 1, 'jacaré': 1, 'k': 1}",Kim K já recebeu a coronavac e está se transfo...
4,"O cidadão diz que a vacina da China faz mal, m...",r2_co,for,no,caiocvm,r2_co_2681,_1337869564290486278,2401,"[('o', 'ART'), ('cidadão', 'N'), ('diz', 'V'),...",[],"{'cidadão': 1, 'distribuição': 1, 'china': 1, ...",O cidadão afirma que a vacina desenvolvida na ...
...,...,...,...,...,...,...,...,...,...,...,...,...
3088,Dps de ter mentido sobre numeros de infectados...,r2_co,for,no,ana_bibi_ana,r2_co_2001,_1340387418294333440,1317,"[('dps', 'N'), ('de', 'PREP'), ('ter', 'V'), (...",[],"{'dps': 1, 'numero': 1, 'aq': 1, 'numeros': 1,...",Após terem mentido sobre os números de infecta...
3089,"Eu acho que quem deveriam ser voluntários, par...",r2_co,against,no,carivaldomelo,r2_co_2744,_1271314875214524423,249,"[('@', 'NUM'), ('cauemacris', 'N'), ('eu', 'PR...","['cauemacris', 'tenente_coimbra', 'carteirorea...","{'alesp': 2, 'deputados': 2, 'tenente_coimbra'...","Na minha opinião, os voluntários para a vacina..."
3090,"Cheio de pó no nariz, todo cagado de corrupção...",r2_co,against,no,AllanPitz1,r2_co_381,_1340960081916145664,1388,"[('@', 'NUM'), ('eduardopaes_', 'N'), ('@', 'P...","['eduardopaes_', 'mjmacul_lima', 'jdoriajr']","{'doria': 1, 'vice': 1, 'mjmacul_lima': 1, 'má...","Coberto de sujeira e envolvido em corrupção, e..."
3091,"A vacina chinesa é a mais fácil de armazenar, ...",r2_co,for,no,acordageorgee,r2_co_1681,_1318933875230388230,2313,"[('a', 'ART'), ('vacina', 'N'), ('chinesa', 'A...",[],"{'graus': 2, 'bolsonaro': 1, 'tipo': 1, 'filho...",A vacina chinesa se destaca por ser a mais fác...


In [40]:
"""Overlap between the author original text and the gerneric version of the text (OpenAI)"""
df_ustance_cino["cinovac-overlap"]  = df_ustance_cino.apply(lambda x: getNgramOverlap(x, 3, "Text", "generated_Text"), axis=1)
df_ustance_cl["cl-overlap"]         = df_ustance_cl.apply(lambda x: getNgramOverlap(x, 3, "Text", "generated_Text"), axis=1)
df_ustance_igreja["igreja-overlap"] = df_ustance_igreja.apply(lambda x: getNgramOverlap(x, 3, "Text", "generated_Text"), axis=1)

print(df_ustance_cino["cinovac-overlap"].mean())
print(df_ustance_cl["cl-overlap"].mean())
print(df_ustance_igreja["igreja-overlap"].mean())


0.33918852316048054
0.3179766293622779
0.3090933708205896


In [39]:
print("Orignal text size:\n")

df_ustance_cino["original-size"] = df_ustance_cino.apply(lambda x: get_size(x["Text"]), axis=1)
df_ustance_cl["original-size"] = df_ustance_cl.apply(lambda x: get_size(x["Text"]), axis=1)
df_ustance_igreja["original-size"] = df_ustance_igreja.apply(lambda x: get_size(x["Text"]), axis=1)



print(df_ustance_cino["original-size"].mean())
print(df_ustance_cl["original-size"].mean())
print(df_ustance_igreja["original-size"].mean())


print("----------------------")
print("OpenAI text size:\n")    

df_ustance_cino["open-size"] = df_ustance_cino.apply(lambda x: get_size(x["generated_Text"]), axis=1)
df_ustance_cl["open-size"] = df_ustance_cl.apply(lambda x: get_size(x["generated_Text"]), axis=1)
df_ustance_igreja["open-size"] = df_ustance_igreja.apply(lambda x: get_size(x["generated_Text"]), axis=1)



print(df_ustance_cino["open-size"].mean())
print(df_ustance_cl["open-size"].mean())
print(df_ustance_igreja["open-size"].mean())

Orignal text size:

29.77012609117362
30.048366013071895
24.696450939457204
----------------------
OpenAI text size:

33.196572906563205
33.659694989106754
27.379958246346554


In [46]:
"""Kendau-tau between the author original text and the gerneric version of the text (OpenAI)"""



df_ustance_cino["cino-kendall"] = df_ustance_cino.apply(lambda x: get_kendall_tau(x, "Text", "generated_Text"), axis=1)
df_ustance_cl["cl-kendall"] = df_ustance_cl.apply(lambda x: get_kendall_tau(x, "Text", "generated_Text"), axis=1)
df_ustance_igreja["igreja-kendall"] = df_ustance_igreja.apply(lambda x: get_kendall_tau(x, "Text", "generated_Text"), axis=1)

print(df_ustance_cino["cino-kendall"].mean())
print(df_ustance_cl["cl-kendall"].mean())
print(df_ustance_igreja["igreja-kendall"].mean())



0.8517823658432843
0.8330229200636634
0.8302863780313764
